# Text Vectorization Techniques

**Goal:** Convert text (words/sentences) into numbers so ML models can process them.

We'll cover:
1. **One-Hot Encoding** - Binary representation of words
2. **Bag of Words (BOW)** - Word frequency counts
3. **TF-IDF** - Weighted word importance

In [ ]:
# Import required libraries
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

---
## 1. One-Hot Encoding

Each word becomes a vector with:
- Length = vocabulary size
- Only ONE position is 1, rest are 0

**Example:** If vocabulary is [cat, dog, bird]
- cat  → [1, 0, 0]
- dog  → [0, 1, 0]
- bird → [0, 0, 1]

In [ ]:
# Step 1: Define your document
document = ["my name is sunny and I love AI"]
print("Original document:", document)

In [ ]:
# Step 2: Tokenize - split into words and lowercase
tokens = [sentence.lower().split() for sentence in document]
print("Tokens:", tokens)

In [ ]:
# Step 3: Prepare data for OneHotEncoder
# OneHotEncoder needs each word as a separate list: [[word1], [word2], ...]
all_words = [[word] for sentence in tokens for word in sentence]
print("Words prepared for encoder:", all_words)

In [ ]:
# Step 4: Create and FIT the encoder
# fit() learns the vocabulary from the data
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(all_words)

print("Vocabulary learned:", encoder.categories_[0])
print("Vocabulary size:", len(encoder.categories_[0]))

In [ ]:
# Step 5: TRANSFORM - convert words to one-hot vectors
for sentence in tokens:
    print("\n--- Sentence:", sentence, "---")
    encoded_sentence = encoder.transform([[word] for word in sentence])
    
    # Show each word and its encoding
    for word, vector in zip(sentence, encoded_sentence):
        print(f"  {word:6} → {vector.astype(int)}")

### Understanding the Output

The vocabulary is sorted alphabetically: `['ai', 'and', 'i', 'is', 'love', 'my', 'name', 'sunny']`

So:
- `my` → position 5 is 1 → `[0, 0, 0, 0, 0, 1, 0, 0]`
- `ai` → position 0 is 1 → `[1, 0, 0, 0, 0, 0, 0, 0]`

---
## 2. Bag of Words (BOW)

Counts word occurrences in each document.

**Example:** "I love love pizza"
- Vocabulary: [i, love, pizza]
- BOW: [1, 2, 1] (love appears twice)

In [ ]:
# Multiple documents for BOW
documents = [
    "people watch movie and watch movie again", 
    "people watch cricket and watch cricket",
    "people like movie and like movie a lot",
    "people like cricket"
]

for i, doc in enumerate(documents):
    print(f"Doc {i+1}: {doc}")

In [ ]:
# Create and fit_transform in one step
bow = CountVectorizer()
bow_matrix = bow.fit_transform(documents)

print("Vocabulary:", bow.get_feature_names_out())
print("\nBOW Matrix shape:", bow_matrix.shape)
print("(4 documents, 8 unique words)")

In [ ]:
# Display the BOW matrix
print("Vocabulary:", list(bow.get_feature_names_out()))
print("\nBOW Vectors:")
print("-" * 60)

for i, (doc, vector) in enumerate(zip(documents, bow_matrix.toarray())):
    print(f"Doc {i+1}: {vector}")
    print(f"        '{doc}'\n")

In [ ]:
# Understanding the counts
vocab = bow.get_feature_names_out()
doc1 = documents[0]  # "people watch movie and watch movie again"
vec1 = bow_matrix.toarray()[0]

print(f"Document: '{doc1}'\n")
print("Word counts:")
for word, count in zip(vocab, vec1):
    if count > 0:
        print(f"  '{word}' appears {count} time(s)")

In [ ]:
# Transform a NEW document (using existing vocabulary)
new_doc = ["people watch cricket movie"]
new_vector = bow.transform(new_doc)

print("New document:", new_doc[0])
print("Vocabulary:", list(vocab))
print("BOW vector:", new_vector.toarray()[0])

In [ ]:
# What about unknown words?
unknown_doc = ["lion is the king of jungle"]
unknown_vector = bow.transform(unknown_doc)

print("Document with unknown words:", unknown_doc[0])
print("BOW vector:", unknown_vector.toarray()[0])
print("\nAll zeros! Words not in vocabulary are ignored.")

---
## 3. TF-IDF (Term Frequency - Inverse Document Frequency)

**Problem with BOW:** Common words like "the", "and", "people" get high counts but aren't meaningful.

**TF-IDF Solution:** 
- Words appearing in MANY documents get LOWER scores
- Words appearing in FEW documents get HIGHER scores

**Formula:**
- TF = word count in document / total words
- IDF = log(total documents / documents with this word)
- TF-IDF = TF × IDF

In [ ]:
# Same documents
documents = [
    "people watch movie and watch movie again", 
    "people watch cricket and watch cricket",
    "people like movie and like movie a lot",
    "people like cricket"
]

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

print("Vocabulary:", list(tfidf.get_feature_names_out()))

In [ ]:
# Display TF-IDF matrix
vocab = tfidf.get_feature_names_out()

print("TF-IDF Vectors (rounded to 2 decimals):")
print("Vocab:", list(vocab))
print("-" * 70)

for i, (doc, vector) in enumerate(zip(documents, tfidf_matrix.toarray())):
    print(f"\nDoc {i+1}: {np.round(vector, 2)}")
    print(f"        '{doc}'")

In [ ]:
# Compare word importance across documents
print("Word importance analysis:\n")

# 'people' appears in ALL documents - should have LOW score
people_idx = list(vocab).index('people')
print(f"'people' (in ALL 4 docs):")
for i, vec in enumerate(tfidf_matrix.toarray()):
    print(f"  Doc {i+1}: {vec[people_idx]:.3f}")

print()

# 'again' appears in only 1 document - should have HIGH score
again_idx = list(vocab).index('again')
print(f"'again' (in only 1 doc):")
for i, vec in enumerate(tfidf_matrix.toarray()):
    print(f"  Doc {i+1}: {vec[again_idx]:.3f}")

In [ ]:
# Find most important words in each document
print("Most important words per document:\n")

for i, (doc, vector) in enumerate(zip(documents, tfidf_matrix.toarray())):
    # Get indices sorted by TF-IDF score (descending)
    sorted_indices = np.argsort(vector)[::-1]
    
    print(f"Doc {i+1}: '{doc}'")
    print("  Top words:", end=" ")
    for idx in sorted_indices[:3]:  # Top 3 words
        if vector[idx] > 0:
            print(f"{vocab[idx]}({vector[idx]:.2f})", end="  ")
    print("\n")

---
## Comparison Summary

| Method | Output | Pros | Cons |
|--------|--------|------|------|
| **One-Hot** | Binary vector per word | Simple | Large vectors, no meaning |
| **BOW** | Word counts per document | Captures frequency | Common words dominate |
| **TF-IDF** | Weighted importance | Highlights unique words | Still ignores word order |

---
## Key Concepts to Remember

### fit() vs transform() vs fit_transform()

- **fit()** - Learn from data (build vocabulary)
- **transform()** - Apply what was learned (convert new data)
- **fit_transform()** - Do both in one step

```python
# Training: fit + transform
vectorizer.fit(training_docs)
training_vectors = vectorizer.transform(training_docs)

# Or in one step:
training_vectors = vectorizer.fit_transform(training_docs)

# Testing: only transform (use same vocabulary)
test_vectors = vectorizer.transform(test_docs)
```

In [ ]:
# Complete workflow example
print("=" * 50)
print("COMPLETE WORKFLOW EXAMPLE")
print("=" * 50)

# Training documents
train_docs = [
    "I love machine learning",
    "deep learning is amazing",
    "python is great for AI"
]

# Test document
test_docs = ["I love python and AI"]

# Create vectorizer and fit on training data
vectorizer = TfidfVectorizer()
train_vectors = vectorizer.fit_transform(train_docs)

print("\nVocabulary learned from training:")
print(list(vectorizer.get_feature_names_out()))

# Transform test data using SAME vocabulary
test_vectors = vectorizer.transform(test_docs)

print("\nTest document:", test_docs[0])
print("Test vector:", np.round(test_vectors.toarray()[0], 2))